## **Important Note: this note is devided into two sections**
**1- Explaination code of every step in the pre-processing step befor creating the pipeline**

**2- The main pipeline to be used for trainig and evaluation**

**`Just run the piplines section, the first section is just to demonstrate our work`**

## **Importing Required Libraries**

In [ ]:
!pip install tnkeeh

In [ ]:
!pip install spark-nlp

In [ ]:

!pip  install -U farasapy

In [ ]:
!pip install joblib

In [ ]:
!pip install swifter

In [ ]:
!pip install swifter dask tqdm psutil

In [ ]:
!pip install nltk

In [ ]:
!pip install light_stem

## **Importing necessary libraries**

In [ ]:
import pandas as pd
import re
import tnkeeh as tn
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords
import sparknlp
from sparknlp.base import DocumentAssembler, LightPipeline
from sparknlp.annotator import Tokenizer, LemmatizerModel
from farasa.stemmer import FarasaStemmer
from pyspark.ml import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from joblib import Parallel, delayed
import multiprocessing
import matplotlib.pyplot as plt
import seaborn as sns

# <font size="6" color="teal"> **Explaination of every step in the pre-processing step befor creating the pipeline**</font>

<font size="4" color="red">Note: Don't run the cells in the explanation</font>

## **Data Loading**

In [ ]:
!git clone https://github.com/Fatma-Alhabbash/Fake-News-detection-Project.git
%cd Fake-News-detection-Project

In [ ]:
url = "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv"

In [ ]:
data = pd.read_csv(url)

In [ ]:
data.head()

## **Data Preprocessing**


### 1. Searching for any null values in the dataset

In [ ]:
# Replace empty strings with NaN
data.replace('', pd.NA, inplace=True)

In [ ]:
# counting the number of missing values in the dataset
data.isnull().sum()

#### Fortunately, there are no null values ​​in the dataset.


## Exploratory Data Analysis (EDA)

**1.Analyze Label Distribution**

In [ ]:
# Define total number of rows
total = len(data)
print(f"Total Number of Articles: {total}")

In [ ]:
# Step 3: Analyze Label Distribution
print("\nStep 3: Label Distribution")
label_counts = data['Label'].str.lower().value_counts()
print("Label Counts:")
print(label_counts)
# Visualize distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='Label', data=data)
plt.title('Distribution of Real vs Fake News')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

**2.Platform Distribution**

In [ ]:
print("\n3.2: Platform Distribution")
platform_counts = data['platform'].value_counts()
print("Platform Counts:")
print(platform_counts)
platform_percentages = (platform_counts / total * 100).round(2)
print("Platform Percentages:")
print(platform_percentages)
# Visualize platform distribution
plt.figure(figsize=(10, 6))
sns.countplot(y='platform', data=data, order=platform_counts.index)
plt.title('Distribution of News by Platform')
plt.xlabel('Count')
plt.ylabel('Platform')
plt.savefig('platform_distribution.png')
plt.show()

**3.Temporal Distribution**

In [ ]:
print("\n3.3: Temporal Distribution")
data['date'] = pd.to_datetime(data['date'], errors='coerce')
data['year_month'] = data['date'].dt.to_period('M')
temporal_counts = data['year_month'].value_counts().sort_index()
print("Article Counts by Year-Month:")
print(temporal_counts)
# Visualize temporal distribution
plt.figure(figsize=(12, 6))
temporal_counts.plot(kind='bar')
plt.title('Temporal Distribution of Articles')
plt.xlabel('Year-Month')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.savefig('temporal_distribution.png')
plt.show()

In [ ]:
print("\n3.7: Correlation Analysis")
data['label_numeric'] = data['Label'].map({'fake': 1, 'real': 0})
correlation = data[['text_length', 'label_numeric']].corr()
print("Correlation Matrix:")
print(correlation)
# Visualize correlation matrix
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Text Features and Label')
plt.savefig('correlation_matrix.png')
plt.show()

### 3. Merging the tilte and the news content columns

In [ ]:

data['content'] = data['title']+' '+data['News content']

In [ ]:
data['content'][1]

### 3. **Cleaning the data** by removing any non-Arabic letters, full URLs, including Twitter links, English words followed by numbers (with or without space), and removing extra whitespace

In [ ]:
cleaner = tn.Tnkeeh(normalize=True)

def clean_text(text):
    if isinstance(text, str):

        # Step 0: Remove # but keep the Arabic word after it (e.g., #غزة → غزة)
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)

        # Step 1: Remove full URLs, including Twitter links
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)

        # Step 1: Remove English words followed by numbers (with or without space)
        text = re.sub(r'\b[A-Za-z]+\s+[0-9]+\b', '', text)
        text = re.sub(r'\b[A-Za-z]+\s+\d+\b', '', text)

        # Step 2: Remove standalone English words
        text = re.sub(r'\b[A-Za-z]+\b', '', text)

        # Step 3: Remove Twitter mentions and other hashtags (non-Arabic only)
        text = re.sub(r'http\S+|www\S+|pic\.twitter\.com/\S+|@\S+', '', text)

        # Step 4: Remove all non-Arabic letters and numbers (but keep Arabic/English digits)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)

        # Step 5: Match digits that have the same writing but different encodings.
        text = cleaner.clean_raw_text(text)[0]

        # Final cleanup: remove extra whitespace
        return re.sub(r'\s+', ' ', text).strip()


    return text


### 4. Removing stop words and lemmatizing or each word

In [ ]:
import requests
url2 = "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
response = requests.get(url2)

stopwords = response.text


#### This stop words file can be found in the following link :
(https://github.com/mohataher/arabic-stop-words?source=post_page-----ba9f1d2e8cb7---------------------------------------)

In [ ]:
def stop_words_removal(text_column, stopwords):
    return text_column.apply(lambda text: ' '.join(
        word for word in str(text).split() if word not in stopwords
    ))


#### a. Stemming using ISRI Stemmer

In [ ]:
def ISRI_stemmer(text, stopwords):
    st = ISRIStemmer()
    result = ''

    for word in text.split():
        if word not in stopwords:
          if len(word) > 1:
            lemma = st.suf32(word)
            result += lemma + ' '

    return result.strip()

In [ ]:
data_copy_ISRI_stemmer = data.copy()

In [ ]:
# Apply clean_text and then lemmatize_arabic_text
data_copy_ISRI_stemmer['content'] = data_copy_ISRI_stemmer['content'].apply(clean_text)

In [ ]:
data_copy_ISRI_stemmer['content'] = data_copy_ISRI_stemmer['content'].apply(lambda x: ISRI_stemmer(x, stopwords))

In [ ]:
data_copy_ISRI_stemmer['content'][0]

In [ ]:
# add the platform column to the content column for prediction
data_copy_ISRI_stemmer['content'] += ' '+data_copy_ISRI_stemmer['platform']

In [ ]:
data_copy_ISRI_stemmer['content'][0]

#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
X = data_copy_ISRI_stemmer['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X)
data_copy_ISRI_stemmer_vectorized = vectorizer.transform(X)

In [ ]:
print(data_copy_ISRI_stemmer_vectorized)

#### Convert the label column to 1 as fake and 0 as real



In [ ]:
# Make sure all label values are lowercase
Y = data_copy_ISRI_stemmer['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y = Y.map({'fake': 1, 'real': 0})

print(Y)

In [ ]:
# prompt: mount google drive

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# change to ur drive link
data_copy_ISRI_stemmer.to_csv("/content/drive/MyDrive/NLP_Project/cleaned_dataset_using_ISRI_stemmer.csv", index=False, encoding='utf-8-sig')
Y.to_csv("/content/drive/MyDrive/NLP_Project/Y_ISRI.csv")

#### b. Lemmatizing using Spark NLP Arabic lemmatizer

In [ ]:
# Set up Spark NLP and pipeline

# Start Spark NLP session
spark = sparknlp.start()

# Build pipeline
document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

lemmatizer = LemmatizerModel.pretrained("lemma", "ar") \
    .setInputCols(["token"]) \
    .setOutputCol("lemma")

nlp_pipeline = Pipeline(stages=[document_assembler, tokenizer, lemmatizer])

# LightPipeline for fast local processing
empty_df = spark.createDataFrame([[""]]).toDF("text")
light_pipeline = LightPipeline(nlp_pipeline.fit(empty_df))


In [ ]:
def spark_lemmatizer(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    results = light_pipeline.fullAnnotate(text)
    lemmas = [lemma.result for lemma in results[0]['lemma']]
    return ' '.join(lemmas)

In [ ]:
data_copy_spark_lemmatizer = data.copy()

In [ ]:
# Apply clean_text and then stop words removal
data_copy_spark_lemmatizer['content'] = data_copy_spark_lemmatizer['content'].apply(clean_text)
data_copy_spark_lemmatizer['content'] = stop_words_removal(data_copy_spark_lemmatizer['content'], stopwords)

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()
data_copy_spark_lemmatizer['content'] = data_copy_spark_lemmatizer['content'].progress_apply(lambda x: spark_lemmatizer(x))

In [ ]:
data_copy_spark_lemmatizer['content'][0]

In [ ]:
data_copy_spark_lemmatizer.to_csv("/cleaned_dataset_using_spark_lemmatizer.csv", index=False, encoding='utf-8-sig')

In [ ]:
# change to ur drive link
spark_lemmatizer_data = pd.read_csv("/content/drive/MyDrive/NLP-project/cleaned_dataset_using_spark_lemmatizer.csv")


#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
# add the platform column to the content column for prediction
spark_lemmatizer_data['content'] += spark_lemmatizer_data['platform']
X2 = spark_lemmatizer_data['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X2)
spark_lemmatizer_data_vectorized = vectorizer.transform(X2)

#### Convert the label column to 1 as fake and 0 as real

In [ ]:
# Make sure all label values are lowercase
Y2 = spark_lemmatizer_data['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y2 = Y2.map({'fake': 1, 'real': 0})

print(Y2)

In [ ]:
# change to ur drive link

spark_lemmatizer_data.to_csv("/content/drive/MyDrive/NLP-project/cleaned_dataset_using_spark_lemmatizer.csv", index=False, encoding='utf-8-sig')
Y2.to_csv("/content/drive/MyDrive/NLP-project/Y_spark.csv")

#### c. Stemming using Farasa
<font size="3" color="red">Need long time to run so we didn't use it</font>

In [ ]:
# stemmer = FarasaStemmer()
# def farasa_stemmer(text):
#   return stemmer.stem(text)

In [ ]:
# data_copy3 = data.copy()

In [ ]:
# data_copy3['content'] = data_copy3['content'].apply(clean_text)
# data_copy3['content'] = stop_words_removal(data_copy3['content'], stopwords)

In [ ]:
# from multiprocessing import Pool, cpu_count
# from tqdm import tqdm

# def stem_text(text):
#     return stemmer.stem(text)

# def parallel_stem(texts, n_cores=None):
#     if n_cores is None:
#         n_cores = cpu_count() - 1  # Leave one core free

#     with Pool(n_cores) as pool:
#         # Use tqdm to show progress
#         results = list(tqdm(pool.imap(stem_text, texts), total=len(texts)))
#     return results
# data_copy3['content'] = parallel_stem(data_copy3['content'].tolist())

#### d. Stemming using light_stem.py

It is a python file developed by Engineer Motaz Saad and here is the link (https://github.com/motazsaad/arabic-light-stemming-py)



In [ ]:
# Download the raw file using wget
!wget https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/light_stem.py

# Now you can import it
import light_stem

In [ ]:
data_copy_light_stem = data.copy()

In [ ]:
data_copy_light_stem['content'] = data_copy_light_stem['content'].apply(clean_text)
data_copy_light_stem['content'] = stop_words_removal(data_copy_light_stem['content'], stopwords)
data_copy_light_stem['content'] = data_copy_light_stem['content'].apply(light_stem.light_stem)

In [ ]:
data_copy_light_stem['content'][0]

#### Convert textual input feature into numerical data using TF-IDF

In [ ]:
# add the platform column to the content column for prediction
data_copy_light_stem['content'] += data_copy_light_stem['platform']
X3 = data_copy_light_stem['content'].values

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X3)
data_copy_light_stem_vectorized = vectorizer.transform(X3)

#### Convert the label column to 1 as fake and 0 as real

In [ ]:
# Make sure all label values are lowercase
Y3 = data_copy_light_stem['Label'].str.lower()

# Convert 'fake' to 1, 'real' to 0
Y3 = Y3.map({'fake': 1, 'real': 0})

print(Y3)

# <font size="6" color="teal"> **The main pipeline to be used for trainig and evaluation**</font>

In [ ]:
## Required libraries for the pipelines
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
import requests
cleaner = tn.Tnkeeh(normalize=True)

## 1. Pipeline for Logistic Regression + ISRI Stemmer

In [ ]:
# --- Load Data ---
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    df['content'] = df['title'] + ' ' + df['News content']
    return df

# --- Clean Arabic Text ---
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)
        text = re.sub(r'\b[A-Za-z]+\s*\d*\b', '', text)
        text = re.sub(r'@\S+', '', text)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)
        text = cleaner.clean_raw_text(text)[0]
        return re.sub(r'\s+', ' ', text).strip()
    return text

# --- Apply ISRI Stemming ---
def ISRI_stem(text, stopwords):
    st = ISRIStemmer()
    return ' '.join([
        st.suf32(word) for word in text.split()
        if word not in stopwords and len(word) > 1
    ])

# --- Load Arabic Stopwords ---
def load_stopwords(raw_url):
    response = requests.get(raw_url)
    response.raise_for_status()  # Raises an error if the request fails
    return set(response.text.strip().splitlines())

# --- Full Preprocessing ---
def preprocess(df, stopwords):
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(lambda x: ISRI_stem(x, stopwords))
    df['content'] += ' ' + df['platform']
    return df

# --- TF-IDF Vectorization ---
def vectorize(df):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['content'])
    y = df['Label'].str.lower().map({'fake': 1, 'real': 0})
    return X, y, vectorizer

# --- Train & Evaluate Model ---
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("\n Model Evaluation:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:\n", classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    return model

# --- Master Function ---
def run_pipeline(data_path, stopwords_path):
    df = load_data(data_path)
    stopwords = load_stopwords(stopwords_path)
    df = preprocess(df, stopwords)
    X, y, vectorizer = vectorize(df)
    model = train_model(X, y)
    return model, vectorizer, stopwords

# --- Predict News ---
def predict_news(news_text, model, vectorizer, stopwords):
    cleaned = clean_text(news_text)
    stemmed = ISRI_stem(cleaned, stopwords)
    tfidf_vector = vectorizer.transform([stemmed])
    prediction = model.predict(tfidf_vector)[0]
    print("\nNews Prediction:", "Ooh it is FAKE news 🤥" if prediction == 1 else "It is REAL news 🫡")

# --- Run ---
model, vectorizer, stopwords = run_pipeline(
    data_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"

)


In [ ]:
input_news = input("Write the news to detect if it is real or fake 🧐: ")
predict_news(input_news, model, vectorizer, stopwords)

## 2. Pipeline for Logistic Regression + Spark NLP Lemmatizer

In [ ]:
from tqdm.notebook import tqdm
tqdm.pandas()

# --- Load Data ---
def load_data(csv_path):
    df = pd.read_csv(csv_path)
    df['content'] = df['title'] + ' ' + df['News content']
    return df

# --- Clean Arabic Text ---
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)
        text = re.sub(r'\b[A-Za-z]+\s*\d*\b', '', text)
        text = re.sub(r'@\S+', '', text)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)
        text = cleaner.clean_raw_text(text)[0]
        return re.sub(r'\s+', ' ', text).strip()
    return text

# --- Start Spark NLP ---
spark = sparknlp.start()

# --- Build Spark NLP pipeline ---
document_assembler = DocumentAssembler().setInputCol("text").setOutputCol("document")
tokenizer = Tokenizer().setInputCols(["document"]).setOutputCol("token")
lemmatizer = LemmatizerModel.pretrained("lemma", "ar").setInputCols(["token"]).setOutputCol("lemma")

pipeline = Pipeline(stages=[document_assembler, tokenizer, lemmatizer])
empty_df = spark.createDataFrame([[""]]).toDF("text")
light_pipeline = LightPipeline(pipeline.fit(empty_df))

# --- Lemmatization function using Spark NLP ---
def spark_lemmatize(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    results = light_pipeline.fullAnnotate(text)
    lemmas = [lemma.result for lemma in results[0]['lemma']]
    return ' '.join(lemmas)

# --- Load Arabic Stopwords ---
def load_stopwords(raw_url):
    response = requests.get(raw_url)
    response.raise_for_status()  # Raises an error if the request fails
    return set(response.text.strip().splitlines())

# --- Full Preprocessing and applying Spark lemmatizer---
def preprocess_with_spark_lemmatizer(df, stopwords):
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(lambda text: ' '.join(
        word for word in str(text).split() if word not in stopwords
    ))
    df['content'] = df['content'].progress_apply(spark_lemmatize)
    df['content'] += ' ' + df['platform']  # Add platform as context
    return df

# --- TF-IDF Vectorization ---
def vectorize(df):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['content'])
    y = df['Label'].str.lower().map({'fake': 1, 'real': 0})
    return X, y, vectorizer

# --- Train & Evaluate Model ---
def train_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("\n Evaluation:")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Classification Report:\n", classification_report(y_test, preds))
    print("Confusion Matrix:\n", confusion_matrix(y_test, preds))
    return model

# --- Master Function ---
def run_pipeline_spark_nlp(data_path, stopwords_path):
    df = load_data(data_path)
    stopwords = load_stopwords(stopwords_path)
    df = preprocess_with_spark_lemmatizer(df, stopwords)
    X, y, vectorizer = vectorize(df)
    model = train_model(X, y)
    return model, vectorizer, stopwords

# --- Predict News ---
def predict_news(text, model, vectorizer, stopwords):
    # Preprocess
    cleaned = clean_text(text)
    cleaned = ' '.join([w for w in cleaned.split() if w not in stopwords])
    lemmatized = spark_lemmatize(cleaned)

    # Vectorize and predict
    X_input = vectorizer.transform([lemmatized])
    prediction = model.predict(X_input)[0]

    print("\nNews Prediction:", "Ooh it is FAKE news 🤥" if prediction == 1 else "It is REAL news 🫡")

# --- Run ---
model, vectorizer, stopwords = run_pipeline_spark_nlp(
    data_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
)


In [ ]:
input_news = input("Write the news to detect if it is real or fake 🧐: ")
predict_news(input_news, model, vectorizer, stopwords)

## 3. Pipeline for Logistic Regression + light_stem Lemmatizer

In [ ]:
# -------------------- 0. Install / import basics --------------------
!pip install -q --upgrade scikit-learn requests

import requests, os, sys, re, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
tqdm.pandas()

# -------------------- 1. Helper: grab a .py file from GitHub RAW --------------------
def download_python_module(raw_url, save_as=None):
    """
    Downloads a raw .py file from GitHub (or any URL) and imports it dynamically.
    Returns the imported module.
    """
    if save_as is None:
        save_as = raw_url.split('/')[-1]          # derive filename from URL
    response = requests.get(raw_url)
    response.raise_for_status()
    with open(save_as, 'w', encoding='utf-8') as f:
        f.write(response.text)
    # add cwd to path and import
    cwd = os.getcwd()
    if cwd not in sys.path:
        sys.path.append(cwd)
    module_name = os.path.splitext(save_as)[0]
    return __import__(module_name)

# -------------------- 2. Download & import light_stem --------------------
LIGHT_STEM_URL = (
    "https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/"
    "raw/refs/heads/main/light_stem.py"
)
light_stem = download_python_module(LIGHT_STEM_URL)  # <- imported module

# -------------------- 3. Stop‑word loader (GitHub RAW) --------------------
def load_stopwords_from_github(raw_url):
    resp = requests.get(raw_url)
    resp.raise_for_status()
    return set(resp.text.strip().splitlines())

# -------------------- 4. Basic Arabic cleaner --------------------
def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'#(?=[\u0621-\u064A])', ' ', text)
        text = re.sub(r'https?://\S+|www\.\S+|pic\.twitter\.com/\S+|t\.co/\S+', '', text)
        text = re.sub(r'\b[A-Za-z]+\s*\d*\b', '', text)
        text = re.sub(r'@\S+', '', text)
        text = re.sub(r'[^\u0621-\u064A0-9\u0660-\u0669\s]', ' ', text)
        text = cleaner.clean_raw_text(text)[0]
        return re.sub(r'\s+', ' ', text).strip()
    return text

# -------------------- 5. Pre‑processing with light_stem --------------------
def preprocess_with_light_stem(df, stopwords):
    df = df.copy()
    df['content'] = df['title'] + ' ' + df['News content']
    df['content'] = df['content'].apply(clean_text)
    df['content'] = df['content'].apply(                                   # remove stop‑words
        lambda txt: ' '.join(w for w in txt.split() if w not in stopwords)
    )
    df['content'] = df['content'].progress_apply(light_stem.light_stem)    # <-- light stemming
    df['content'] += ' ' + df['platform']                                  # add platform as extra token
    return df

# -------------------- 6. Modelling helper --------------------
def train_evaluate(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print("🔎 Accuracy:", accuracy_score(y_test, preds))
    print("\nClassification report:\n", classification_report(y_test, preds))
    print("Confusion matrix:\n", confusion_matrix(y_test, preds))

# -------------------- 7. Master pipeline --------------------
def run_pipeline_light_stem(data_csv_url_or_path,
                            stopwords_raw_url):
    # a. load data  (works with local path OR http(s)://)
    df = pd.read_csv(data_csv_url_or_path)
    print("Dataset shape:", df.shape)

    # b. stopwords
    stopwords = load_stopwords_from_github(stopwords_raw_url)

    # c. preprocess
    df = preprocess_with_light_stem(df, stopwords)

    # d. TF‑IDF
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(df['content'])
    y = df['Label'].str.lower().map({'fake': 1, 'real': 0})

    # e. train & evaluate
    train_evaluate(X, y)


# -------------------- 8. RUN --------------------
run_pipeline_light_stem(
    data_csv_url_or_path="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/merged_cleaned.csv",
    stopwords_raw_url="https://github.com/Fatma-Alhabbash/Fake-News-detection-Project/raw/refs/heads/main/arabic_stopwords.txt"
)
